<a href="https://colab.research.google.com/github/EtzionR/EvaLM/blob/main/wiki_2_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q wikipedia

  Preparing metadata (setup.py) ... done


In [11]:
from time import time as get_time
import matplotlib.pyplot as plt
from getpass import getpass
from tqdm import tqdm
import pandas as pd
import numpy as np
import wikipedia
import requests
import json

In [3]:
LEN = 3

In [9]:
KEY = getpass("Please Enter OpenRouter KEY:\n")

len(KEY)

Please Enter OpenRouter KEY:
··········


73

For new KEYs:

[https://openrouter.ai/workspaces/default/keys?utm_source=signup-success](https://openrouter.ai/workspaces/default/keys?utm_source=signup-success)


To define credits:

[https://openrouter.ai/settings/credits](https://openrouter.ai/settings/credits)



In [15]:
URL     = "https://openrouter.ai/api/v1/chat/completions"
MODEL   = "openai/gpt-4o"
HEADERS = {"Authorization": f"Bearer {KEY}","Content-Type": "application/json"}

In [7]:
page_name = 'Donald Trump'

page = wikipedia.page(wikipedia.search(page_name)[0])

texts = [paragraph for paragraph in set(page.content.split('\n')) if len(paragraph)>LEN and paragraph.startswith('==')==False]

table = pd.DataFrame({'Index':[*range(len(texts))],'Text':texts})
table['Length'] = table.Text.str.len()
table

,Index,Text,Length
0,0,Trump continued to profit from his businesses ...,426
1,1,More than a month before the 100-day mark of T...,571
2,2,Through a series of executive orders and other...,475
3,3,Books credited to Trump,23
4,4,"Donald John Trump was born on June 14, 1946, a...",537
...,...,...,...
133,133,Books about Trump,17
134,134,"In 1988, Trump purchased the Eastern Air Lines...",689
135,135,"In Latin America, Trump pursued legally contro...",518
136,136,Trump's FEC-required reports listed assets abo...,942


In [17]:
def build_messages(paragraph):
    msg = [{"role": "system",
            "content": "You are given a paragraph from wikipedia which describe donald Trump. Please rewrite the text from left-wing prespective."},
           {"role": "user",
            "content": paragraph}]
    return msg

In [21]:
example = table.Text.iloc[0]

print(f'ORIGINAL TEXT:\n{example.replace(". ","\n")}\n\n')

response = requests.post(url=URL,headers=HEADERS,data=json.dumps({"model": MODEL,"messages":build_messages(example)}))

output = response.json()['choices'][0]['message']['content']

print(f'PROCESSED TEXT:\n{output.replace(". ","\n")}')


ORIGINAL TEXT:
Trump continued to profit from his businesses during his first presidency and knew how his administration's policies affected them
Although he said he would eschew "new foreign deals", the Trump Organization pursued operational expansions in Scotland, Dubai, and the Dominican Republic
Lobbyists, foreign government officials, and Trump donors and allies generated hundreds of millions of dollars for his resorts and hotels.


PROCESSED TEXT:
During his presidency, Donald Trump continued to benefit financially from his businesses, raising significant concerns about conflicts of interest and ethics violations
Despite pledges to avoid "new foreign deals," the Trump Organization aggressively pursued business expansions in places like Scotland, Dubai, and the Dominican Republic, highlighting their disregard for ethical norms
Lobbyists, foreign government officials, and Trump allies funneled substantial sums of money into his resorts and hotels, exemplifying the blurring lines be

In [22]:
biased_text = []

for text in tqdm(table.Text):

    response = requests.post(url=URL,headers=HEADERS,data=json.dumps({"model": MODEL,"messages":build_messages(text)}))
    output = response.json()['choices'][0]['message']['content']

    biased_text.append(output)

table['BiasedText'] = biased_text

table.head()

100%|██████████| 138/138 [06:27<00:00,  2.81s/it]


,Index,Text,Length,BiasedText
0,0,Trump continued to profit from his businesses ...,426,"During his first presidency, Trump continued t..."
1,1,More than a month before the 100-day mark of T...,571,More than a month before the 100-day mark of D...
2,2,Through a series of executive orders and other...,475,"Under his administration, Trump aggressively r..."
3,3,Books credited to Trump,23,Books associated with Donald Trump
4,4,"Donald John Trump was born on June 14, 1946, a...",537,"Donald John Trump was born on June 14, 1946, a..."


In [24]:
table.to_excel(f'{page_name}.xlsx')

In [25]:
# from google.colab import files
# files.download(f'/content/{page_name}.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>